# LSEG `lseg.data` Theme Basket AVWAP flow × position

ユーザ定義のテーマバスケット（構成銘柄とウェイトを所与）を分析単位にして、LSEG Data Library for Python (`lseg.data`) から日次価格・出来高・売買代金を取得し、以下を計算します。

- 銘柄別の年初来アンカー AVWAP
- バスケット別の加重 AVWAP 上回り比率 = position
- バスケット別の売買代金フロー変化 = flow
- `flow × position` の4象限散布図
- バスケット別・銘柄別ドリルダウン

実行前に編集する箇所は、原則として **Configuration** と **basket_constituents** の2セルだけです。

注意: LSEGデータの保存・共有・再配布は契約条件に従ってください。このNotebookはローカル分析用途を想定しています。

## 1. Install & imports

`lseg-data` が未導入の場合は、下の `%pip install` 行のコメントアウトを外して実行してください。

In [ ]:
# %pip install -U lseg-data pandas numpy plotly

from __future__ import annotations

import hashlib
import json
import os
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError("plotly が必要です。`%pip install plotly` を実行してください。") from exc

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.options.mode.copy_on_write = True
warnings.filterwarnings("default")


## 2. Configuration

主にここを編集します。

- `RUN_WITH_MOCK_DATA = False` のままなら LSEG に接続します。
- LSEG接続の認証情報はNotebookに直書きせず、`lseg-data.config.json` または Workspace/Eikon Desktop 経由で管理してください。
- `SESSION_NAME` は通常 `desktop.workspace` または `platform.ldp` です。既定設定を使う場合は `None` にできます。

In [ ]:
# ------------------------------------------------------------
# Runtime mode
# ------------------------------------------------------------
RUN_WITH_MOCK_DATA = os.getenv("RUN_WITH_MOCK_DATA", "0") == "1"  # TrueにするとLSEGへ接続せずサンプルデータで検証
AUTO_CLOSE_SESSION = True

# ------------------------------------------------------------
# LSEG session
# ------------------------------------------------------------
SESSION_NAME = "desktop.workspace"      # None, "desktop.workspace", "platform.ldp", etc.
SESSION_CONFIG_NAME = None              # 例: r"C:\path\to\lseg-data.config.json"。通常は None。
HTTP_REQUEST_TIMEOUT_SECONDS = 120

# ------------------------------------------------------------
# Dates
# ------------------------------------------------------------
AS_OF_DATE = None                       # Noneなら取得データからカバレッジの良い最新日を自動選択
END_DATE = None                         # Noneならローカル日付の今日
ANCHOR_DATE = f"{pd.Timestamp.today().year}-01-01"
LOOKBACK_BUFFER_DAYS = 150              # flow window取得用。ANCHOR_DATEより前が必要な場合に使う
AS_OF_MIN_COVERAGE = 0.80               # AS_OF_DATE自動判定時の最低カバレッジ
MAX_STALE_DAYS_FOR_POSITION = 5         # 評価日当日の価格欠損を直近データで許容する日数

# ------------------------------------------------------------
# LSEG fields
# ------------------------------------------------------------
PRICE_FIELD = "TRDPRC_1"
VOLUME_FIELD = "ACVOL_UNS"
TURNOVER_FIELD = "TRNOVR_UNS"
VWAP_FIELD = "VWAP"

HISTORY_FIELDS = [PRICE_FIELD, VOLUME_FIELD, TURNOVER_FIELD, VWAP_FIELD]
HISTORY_INTERVAL = "daily"

# get_history の adjustments は環境・権限・商品に依存して使えない組合せがあり得ます。
# エラー時は ["exchangeCorrection", "manualCorrection"] または "unadjusted" に変更して再実行してください。
ADJUSTMENTS = ["exchangeCorrection", "manualCorrection", "CCH", "CRE", "RTS"]

STATIC_FIELDS = [
    "TR.CommonName",
    "TR.ISIN",
    "TR.ExchangeName",
    "TR.Currency",
    "TR.CompanyMarketCap",
]

# ------------------------------------------------------------
# Basket position / flow settings
# ------------------------------------------------------------
POSITION_WEIGHT_MODE = "portfolio_weight"  # "portfolio_weight" or "equal_weight"

FLOW_TURNOVER_MODE = "normalized_weighted" # "normalized_weighted", "raw_weighted", "equal_member"
FLOW_DENOMINATOR_MODE = "basket_union"      # "basket_union", "market_universe", "none"
PLOT_X_METRIC = "flow_share_rel_change_pct" # "flow_share_rel_change_pct" or "flow_turnover_rel_change_pct"

FLOW_SHORT_WINDOW = 5
FLOW_BASE_WINDOW = 60
FLOW_BASE_EXCLUDE_RECENT = True

MIN_BASKET_NAMES = 3
MIN_WEIGHT_COVERAGE = 0.80
MIN_VALID_DAYS = 20

# market_universe を分母にしたい場合のみ指定。例: TOPIX構成銘柄など。
MARKET_UNIVERSE_RICS: list[str] = []

# ------------------------------------------------------------
# Retrieval / cache / output
# ------------------------------------------------------------
BATCH_SIZE = 25
SLEEP_SECONDS = 0.35
MAX_RETRIES = 3
USE_CACHE = True
OUTPUT_DIR = Path("./outputs")
CACHE_DIR = OUTPUT_DIR / "cache"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 3. User-defined basket constituents

ここに任意のテーマバスケットを定義します。

必須列:

- `basket_id`
- `basket_name`
- `ric`
- `weight`

同一RICが複数バスケットに入ることを許容します。`weight` は合計1でなくても構いません。計算時にバスケット内で正規化します。

In [ ]:
basket_constituents = pd.DataFrame([
    # 半導体・ハードAI 例
    {"basket_id": "semis_ai", "basket_name": "半導体・ハードAI", "basket_group": "AI・半導体", "ric": "8035.T", "weight": 0.20, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "東京エレクトロン"},
    {"basket_id": "semis_ai", "basket_name": "半導体・ハードAI", "basket_group": "AI・半導体", "ric": "6857.T", "weight": 0.15, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "アドバンテスト"},
    {"basket_id": "semis_ai", "basket_name": "半導体・ハードAI", "basket_group": "AI・半導体", "ric": "6723.T", "weight": 0.15, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "ルネサス"},
    {"basket_id": "semis_ai", "basket_name": "半導体・ハードAI", "basket_group": "AI・半導体", "ric": "4063.T", "weight": 0.10, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "信越化学"},
    {"basket_id": "semis_ai", "basket_name": "半導体・ハードAI", "basket_group": "AI・半導体", "ric": "6146.T", "weight": 0.10, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "ディスコ"},

    # 銀行 例
    {"basket_id": "banks", "basket_name": "銀行", "basket_group": "金融", "ric": "8306.T", "weight": 0.30, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "三菱UFJ"},
    {"basket_id": "banks", "basket_name": "銀行", "basket_group": "金融", "ric": "8316.T", "weight": 0.25, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "三井住友FG"},
    {"basket_id": "banks", "basket_name": "銀行", "basket_group": "金融", "ric": "8411.T", "weight": 0.25, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "みずほFG"},
    {"basket_id": "banks", "basket_name": "銀行", "basket_group": "金融", "ric": "7182.T", "weight": 0.10, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "ゆうちょ銀行"},
    {"basket_id": "banks", "basket_name": "銀行", "basket_group": "金融", "ric": "8309.T", "weight": 0.10, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "三井住友トラスト"},

    # 電力・重電 例
    {"basket_id": "power_heavy", "basket_name": "電力・重電", "basket_group": "インフラ", "ric": "6501.T", "weight": 0.20, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "日立"},
    {"basket_id": "power_heavy", "basket_name": "電力・重電", "basket_group": "インフラ", "ric": "6503.T", "weight": 0.20, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "三菱電機"},
    {"basket_id": "power_heavy", "basket_name": "電力・重電", "basket_group": "インフラ", "ric": "7011.T", "weight": 0.20, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "三菱重工"},
    {"basket_id": "power_heavy", "basket_name": "電力・重電", "basket_group": "インフラ", "ric": "9501.T", "weight": 0.20, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "東京電力HD"},
    {"basket_id": "power_heavy", "basket_name": "電力・重電", "basket_group": "インフラ", "ric": "9503.T", "weight": 0.20, "weight_type": "portfolio", "side": 1, "valid_from": ANCHOR_DATE, "valid_to": None, "note": "関西電力"},
])

basket_constituents


## 4. Helper functions

In [ ]:
def _as_date(value: Any) -> Optional[pd.Timestamp]:
    """Convert a value to normalized pandas Timestamp. Return None for blank/NA."""
    if value is None or (isinstance(value, float) and np.isnan(value)) or pd.isna(value):
        return None
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return None
    return pd.Timestamp(ts).normalize()


def _today_date() -> pd.Timestamp:
    return pd.Timestamp(datetime.now().date())


def _hash_payload(payload: Any) -> str:
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


def chunked(items: list[str], size: int) -> Iterable[list[str]]:
    for i in range(0, len(items), size):
        yield items[i : i + size]


def weighted_median(values: pd.Series, weights: pd.Series) -> float:
    """Weighted median with NA handling."""
    v = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    mask = v.notna() & w.notna() & (w > 0)
    if not mask.any():
        return np.nan
    v = v[mask].to_numpy(dtype=float)
    w = w[mask].to_numpy(dtype=float)
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    cutoff = 0.5 * w.sum()
    return float(v[np.searchsorted(np.cumsum(w), cutoff, side="left")])


def safe_divide(numer: float, denom: float) -> float:
    if denom is None or pd.isna(denom) or denom == 0:
        return np.nan
    return numer / denom


def pct_change_ratio(short_value: float, base_value: float) -> float:
    if pd.isna(short_value) or pd.isna(base_value) or base_value <= 0:
        return np.nan
    return 100.0 * (short_value / base_value - 1.0)


def coalesce_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    lookup = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


def to_list_unique(values: Iterable[Any]) -> list[str]:
    return sorted({str(x).strip() for x in values if pd.notna(x) and str(x).strip()})


## 5. Basket validation

In [ ]:
def validate_basket_constituents(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Validate and standardize user-defined basket constituents."""
    required = ["basket_id", "basket_name", "ric", "weight"]
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise ValueError(f"basket_constituents に必須列がありません: {missing}")

    df = raw.copy()
    optional_defaults = {
        "basket_group": "未分類",
        "weight_type": "portfolio",
        "side": 1,
        "valid_from": None,
        "valid_to": None,
        "display_order": np.nan,
        "note": "",
    }
    for col, default in optional_defaults.items():
        if col not in df.columns:
            df[col] = default

    for col in ["basket_id", "basket_name", "basket_group", "ric", "weight_type", "note"]:
        df[col] = df[col].astype("string").str.strip()

    df["weight"] = pd.to_numeric(df["weight"], errors="coerce")
    df["side"] = pd.to_numeric(df["side"], errors="coerce").fillna(1.0)
    df["valid_from"] = df["valid_from"].apply(_as_date)
    df["valid_to"] = df["valid_to"].apply(_as_date)

    issues: list[dict[str, Any]] = []

    def add_issue(level: str, check: str, detail: str, n: Optional[int] = None) -> None:
        issues.append({"level": level, "check": check, "detail": detail, "count": n})

    for col in ["basket_id", "basket_name", "ric"]:
        blank = (df[col].astype("string").str.len() == 0).fillna(False)
        n = int(df[col].isna().sum() + blank.sum())
        if n:
            add_issue("ERROR", f"missing_{col}", f"{col} が空の行があります", n)

    n_bad_weight = int(df["weight"].isna().sum() + (df["weight"] < 0).sum())
    if n_bad_weight:
        add_issue("ERROR", "bad_weight", "weight は非負の数値である必要があります", n_bad_weight)

    bad_date = df["valid_from"].notna() & df["valid_to"].notna() & (df["valid_from"] > df["valid_to"])
    if bad_date.any():
        add_issue("ERROR", "bad_valid_period", "valid_from > valid_to の行があります", int(bad_date.sum()))

    if not (df["side"].isin([1, 1.0])).all():
        add_issue("ERROR", "unsupported_side", "v1はlong-onlyです。side は 1 にしてください", int((~df["side"].isin([1, 1.0])).sum()))

    dup_cols = ["basket_id", "ric", "valid_from", "valid_to"]
    dup = df.duplicated(dup_cols, keep=False)
    if dup.any():
        add_issue("ERROR", "duplicate_basket_ric", f"同一 {dup_cols} の重複があります", int(dup.sum()))

    weight_sum = df.groupby("basket_id", dropna=False)["weight"].sum(min_count=1)
    zero_sum = weight_sum[weight_sum <= 0]
    if not zero_sum.empty:
        add_issue("ERROR", "zero_weight_sum", f"weight合計が0以下のbasket_id: {list(zero_sum.index)}", len(zero_sum))

    basket_counts = df.groupby("basket_id")["ric"].nunique()
    few = basket_counts[basket_counts < MIN_BASKET_NAMES]
    if not few.empty:
        add_issue("WARN", "few_names", f"構成銘柄数が MIN_BASKET_NAMES 未満: {few.to_dict()}", len(few))

    weight_deviation = (weight_sum - 1.0).abs()
    dev = weight_deviation[weight_deviation > 0.10]
    if not dev.empty:
        add_issue("WARN", "weight_sum_not_one", "weight合計が1から10%以上乖離しています。計算時に正規化します", len(dev))

    concentration = df.assign(weight_share=df["weight"] / df.groupby("basket_id")["weight"].transform("sum"))
    concentrated = concentration[concentration["weight_share"] > 0.50]
    if not concentrated.empty:
        add_issue("WARN", "concentrated_weight", "単一銘柄ウェイトが50%を超えるバスケット構成があります", int(concentrated.shape[0]))

    report = pd.DataFrame(issues, columns=["level", "check", "detail", "count"])
    if any(x["level"] == "ERROR" for x in issues):
        display(report)
        raise ValueError("basket_constituents の検証でERRORが発生しました。上表を修正してください。")

    df["valid_from"] = pd.to_datetime(df["valid_from"])
    df["valid_to"] = pd.to_datetime(df["valid_to"])
    df["display_order"] = pd.to_numeric(df["display_order"], errors="coerce")
    df = df.sort_values(["display_order", "basket_group", "basket_id", "ric"], na_position="last").reset_index(drop=True)
    return df, report


basket_constituents, basket_validation_report = validate_basket_constituents(basket_constituents)
print(f"Baskets: {basket_constituents['basket_id'].nunique():,}, basket × RIC rows: {len(basket_constituents):,}, unique RICs: {basket_constituents['ric'].nunique():,}")
display(basket_validation_report if not basket_validation_report.empty else pd.DataFrame([{"level": "OK", "check": "basket_validation", "detail": "No validation issues", "count": 0}]))
display(basket_constituents.head(20))


## 6. LSEG session and retrieval functions

In [ ]:
def open_lseg_session():
    """Open an LSEG Data Library session. Credentials must be managed outside this notebook."""
    try:
        import lseg.data as ld
    except ImportError as exc:
        raise ImportError("lseg.data が見つかりません。`%pip install -U lseg-data` を実行してください。") from exc

    try:
        try:
            cfg = ld.get_config()
            if hasattr(cfg, "set_param"):
                cfg.set_param("http.request-timeout", HTTP_REQUEST_TIMEOUT_SECONDS)
            else:
                cfg["http.request-timeout"] = HTTP_REQUEST_TIMEOUT_SECONDS
        except Exception as config_exc:
            warnings.warn(f"HTTP timeout設定をスキップしました: {config_exc}")

        kwargs = {}
        if SESSION_NAME:
            kwargs["name"] = SESSION_NAME
        if SESSION_CONFIG_NAME:
            kwargs["config_name"] = SESSION_CONFIG_NAME
        session = ld.open_session(**kwargs)
        print(f"LSEG session opened. SESSION_NAME={SESSION_NAME!r}, CONFIG={SESSION_CONFIG_NAME!r}")
        return ld, session
    except Exception as exc:
        raise RuntimeError(
            "LSEG session open に失敗しました。Workspace/Eikon Desktopが起動しているか、"
            "または platform 用 lseg-data.config.json が正しく設定されているか確認してください。"
        ) from exc


def close_lseg_session(ld: Any) -> None:
    try:
        ld.close_session()
        print("LSEG session closed.")
    except Exception as exc:
        warnings.warn(f"LSEG session close に失敗しました: {exc}")


def get_required_rics(baskets: pd.DataFrame) -> list[str]:
    rics = to_list_unique(baskets["ric"])
    if FLOW_DENOMINATOR_MODE == "market_universe":
        if not MARKET_UNIVERSE_RICS:
            raise ValueError("FLOW_DENOMINATOR_MODE='market_universe' の場合、MARKET_UNIVERSE_RICS を指定してください。")
        rics = sorted(set(rics).union(to_list_unique(MARKET_UNIVERSE_RICS)))
    return rics


def determine_history_dates() -> tuple[pd.Timestamp, pd.Timestamp]:
    end = _as_date(END_DATE) or _today_date()
    anchor = _as_date(ANCHOR_DATE)
    if anchor is None:
        raise ValueError("ANCHOR_DATE を日付として解釈できません。")
    flow_start = end - pd.Timedelta(days=LOOKBACK_BUFFER_DAYS)
    start = min(anchor, flow_start)
    return pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize()


In [ ]:
def standardize_static_metadata(raw: pd.DataFrame, requested_rics: list[str]) -> pd.DataFrame:
    if raw is None or raw.empty:
        return pd.DataFrame({"ric": requested_rics})
    df = raw.copy()
    df.columns = [str(c) for c in df.columns]

    ric_col = coalesce_column(df, ["Instrument", "instrument", "RIC", "TR.RIC", "TRD_RIC"])
    if ric_col is None and len(df.index) == len(df):
        df = df.reset_index()
        ric_col = coalesce_column(df, ["Instrument", "instrument", "index"])
    if ric_col is None:
        values = requested_rics[: len(df)] + [None] * max(0, len(df) - len(requested_rics))
        df.insert(0, "ric", values)
    else:
        df = df.rename(columns={ric_col: "ric"})

    rename_map = {
        "TR.CommonName": "company_name",
        "Company Common Name": "company_name",
        "Common Name": "company_name",
        "TR.ISIN": "isin",
        "ISIN": "isin",
        "TR.ExchangeName": "exchange_name",
        "Exchange Name": "exchange_name",
        "TR.Currency": "currency",
        "Currency": "currency",
        "TR.CompanyMarketCap": "company_market_cap",
        "Company Market Cap": "company_market_cap",
        "Market Capitalization": "company_market_cap",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    if "ric" not in df.columns:
        df["ric"] = np.nan
    df["ric"] = df["ric"].astype("string").str.strip()
    keep = ["ric", "company_name", "isin", "exchange_name", "currency", "company_market_cap"]
    for col in keep:
        if col not in df.columns:
            df[col] = np.nan
    return df[keep].drop_duplicates("ric", keep="first").reset_index(drop=True)


def fetch_static_metadata(ld: Any, rics: list[str]) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    cache_key = _hash_payload({"kind": "static", "rics": rics, "fields": STATIC_FIELDS})
    cache_path = CACHE_DIR / f"static_{cache_key}.pkl"
    if USE_CACHE and cache_path.exists():
        return pd.read_pickle(cache_path), []

    frames: list[pd.DataFrame] = []
    failures: list[dict[str, Any]] = []

    for batch_no, batch in enumerate(chunked(rics, BATCH_SIZE), start=1):
        ok = False
        last_exc = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                raw = ld.get_data(universe=batch, fields=STATIC_FIELDS, use_field_names_in_headers=True)
                frames.append(standardize_static_metadata(raw, batch))
                ok = True
                break
            except Exception as exc:
                last_exc = exc
                time.sleep(SLEEP_SECONDS * attempt)
        if not ok:
            failures.append({"kind": "static", "batch_no": batch_no, "rics": batch, "error": repr(last_exc)})
            frames.append(pd.DataFrame({"ric": batch}))
        time.sleep(SLEEP_SECONDS)

    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame({"ric": []})
    out = standardize_static_metadata(out, rics)
    if USE_CACHE:
        out.to_pickle(cache_path)
    return out, failures


In [ ]:
CANONICAL_FIELD_MAP = {
    PRICE_FIELD: "close",
    VOLUME_FIELD: "volume",
    TURNOVER_FIELD: "turnover_raw",
    VWAP_FIELD: "daily_vwap_raw",
    "Trade Price": "close",
    "Last Price": "close",
    "Close Price": "close",
    "Volume": "volume",
    "Accumulated Volume": "volume",
    "Turnover": "turnover_raw",
    "VWAP": "daily_vwap_raw",
}


def _canonical_field_name(x: Any) -> str:
    s = str(x).strip()
    return CANONICAL_FIELD_MAP.get(s, s)


def normalize_history(raw: pd.DataFrame, requested_rics: list[str]) -> pd.DataFrame:
    """Normalize ld.get_history output to long format: date, ric, close, volume, turnover_raw, daily_vwap_raw."""
    out_cols = ["date", "ric", "close", "volume", "turnover_raw", "daily_vwap_raw"]
    if raw is None or raw.empty:
        return pd.DataFrame(columns=out_cols)

    df = raw.copy()
    if isinstance(df.columns, pd.MultiIndex):
        idx_name = df.index.name or "date"
        tmp = df.copy()
        tmp.index.name = idx_name
        stacked = tmp.stack(list(range(tmp.columns.nlevels)), dropna=False).reset_index()
        value_col = stacked.columns[-1]
        level_cols = list(stacked.columns[1:-1])

        requested_set = set(requested_rics)
        requested_fields = set(CANONICAL_FIELD_MAP.keys()).union(set(HISTORY_FIELDS))
        score = []
        for col in level_cols:
            vals = set(stacked[col].dropna().astype(str).unique())
            score.append((col, len(vals.intersection(requested_set)), len(vals.intersection(requested_fields))))
        ric_col = max(score, key=lambda x: x[1])[0]
        field_col = max(score, key=lambda x: x[2])[0]
        if ric_col == field_col and len(level_cols) >= 2:
            ric_col, field_col = level_cols[0], level_cols[1]

        long = stacked.rename(columns={idx_name: "date", ric_col: "ric", field_col: "field", value_col: "value"})
        long = long[["date", "ric", "field", "value"]]
        long["field"] = long["field"].apply(_canonical_field_name)
        pivot = long.pivot_table(index=["date", "ric"], columns="field", values="value", aggfunc="first").reset_index()
        pivot.columns = [str(c) for c in pivot.columns]
    else:
        df.columns = [str(c) for c in df.columns]
        date_col = coalesce_column(df, ["Date", "DATE", "date", "Timestamp", "TIMESTAMP"])
        ric_col = coalesce_column(df, ["Instrument", "instrument", "RIC", "ric"])

        if date_col is not None:
            pivot = df.rename(columns={date_col: "date"})
            if ric_col is not None:
                pivot = pivot.rename(columns={ric_col: "ric"})
            elif len(requested_rics) == 1:
                pivot["ric"] = requested_rics[0]
            else:
                pivot["ric"] = requested_rics[0] if requested_rics else np.nan
        else:
            pivot = df.reset_index().rename(columns={df.index.name or "index": "date"})
            if len(requested_rics) == 1:
                pivot["ric"] = requested_rics[0]
            elif "Instrument" in pivot.columns:
                pivot = pivot.rename(columns={"Instrument": "ric"})
            else:
                pivot["ric"] = requested_rics[0] if requested_rics else np.nan

        pivot = pivot.rename(columns={c: _canonical_field_name(c) for c in pivot.columns})
        if "date" not in pivot.columns and "Date" in pivot.columns:
            pivot = pivot.rename(columns={"Date": "date"})
        if "ric" not in pivot.columns and "Instrument" in pivot.columns:
            pivot = pivot.rename(columns={"Instrument": "ric"})

    for col in out_cols:
        if col not in pivot.columns:
            pivot[col] = np.nan

    result = pivot[out_cols].copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.normalize()
    result["ric"] = result["ric"].astype("string").str.strip()
    for col in ["close", "volume", "turnover_raw", "daily_vwap_raw"]:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result = result.dropna(subset=["date", "ric"])
    result = result.drop_duplicates(["date", "ric"], keep="last").sort_values(["ric", "date"]).reset_index(drop=True)
    return result


def finalize_history_fields(hist: pd.DataFrame) -> pd.DataFrame:
    """Build clean close/volume/turnover/daily_vwap columns with fallback rules."""
    df = hist.copy()
    for col in ["close", "volume", "turnover_raw", "daily_vwap_raw"]:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df.loc[df["close"] <= 0, "close"] = np.nan
    df.loc[df["volume"] <= 0, "volume"] = np.nan
    df.loc[df["turnover_raw"] <= 0, "turnover_raw"] = np.nan
    df.loc[df["daily_vwap_raw"] <= 0, "daily_vwap_raw"] = np.nan

    turnover_from_vwap = df["daily_vwap_raw"] * df["volume"]
    turnover_from_close = df["close"] * df["volume"]

    df["turnover"] = df["turnover_raw"]
    df["turnover_source"] = np.where(df["turnover"].notna(), TURNOVER_FIELD, pd.NA)

    use_vwap = df["turnover"].isna() & turnover_from_vwap.notna()
    df.loc[use_vwap, "turnover"] = turnover_from_vwap[use_vwap]
    df.loc[use_vwap, "turnover_source"] = f"{VWAP_FIELD}_x_{VOLUME_FIELD}"

    use_close = df["turnover"].isna() & turnover_from_close.notna()
    df.loc[use_close, "turnover"] = turnover_from_close[use_close]
    df.loc[use_close, "turnover_source"] = f"{PRICE_FIELD}_x_{VOLUME_FIELD}"

    df["daily_vwap"] = df["daily_vwap_raw"]
    use_turnover_vwap = df["daily_vwap"].isna() & df["turnover"].notna() & df["volume"].notna() & (df["volume"] > 0)
    df.loc[use_turnover_vwap, "daily_vwap"] = df.loc[use_turnover_vwap, "turnover"] / df.loc[use_turnover_vwap, "volume"]

    df.loc[df["turnover"] <= 0, "turnover"] = np.nan
    return df[["date", "ric", "close", "volume", "turnover", "daily_vwap", "turnover_source"]]


def fetch_history_batched(ld: Any, rics: list[str], start: pd.Timestamp, end: pd.Timestamp) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    payload = {"kind": "history", "rics": rics, "fields": HISTORY_FIELDS, "interval": HISTORY_INTERVAL, "start": str(start.date()), "end": str(end.date()), "adjustments": ADJUSTMENTS}
    cache_key = _hash_payload(payload)
    cache_path = CACHE_DIR / f"history_{cache_key}.pkl"
    if USE_CACHE and cache_path.exists():
        return pd.read_pickle(cache_path), []

    frames: list[pd.DataFrame] = []
    failures: list[dict[str, Any]] = []

    for batch_no, batch in enumerate(chunked(rics, BATCH_SIZE), start=1):
        ok = False
        last_exc = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                raw = ld.get_history(
                    universe=batch,
                    fields=HISTORY_FIELDS,
                    interval=HISTORY_INTERVAL,
                    start=str(start.date()),
                    end=str(end.date()),
                    adjustments=ADJUSTMENTS,
                    use_field_names_in_headers=True,
                )
                frames.append(normalize_history(raw, batch))
                ok = True
                break
            except Exception as exc:
                last_exc = exc
                time.sleep(SLEEP_SECONDS * attempt)
        if not ok:
            failures.append({"kind": "history", "batch_no": batch_no, "rics": batch, "error": repr(last_exc)})
        time.sleep(SLEEP_SECONDS)

    hist = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["date", "ric", "close", "volume", "turnover_raw", "daily_vwap_raw"])
    hist = finalize_history_fields(hist)
    hist = hist.drop_duplicates(["date", "ric"], keep="last").sort_values(["ric", "date"]).reset_index(drop=True)
    if USE_CACHE:
        hist.to_pickle(cache_path)
    return hist, failures


## 7. Calculation functions

In [ ]:
def generate_mock_history(rics: list[str], start: pd.Timestamp, end: pd.Timestamp, seed: int = 7) -> pd.DataFrame:
    """Generate deterministic sample data for offline smoke testing."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    rows = []
    for ric in rics:
        n = len(dates)
        base = rng.uniform(800, 6000)
        rets = rng.normal(0.0005, 0.018, size=n)
        close = base * np.exp(np.cumsum(rets))
        volume = rng.lognormal(mean=13.0, sigma=0.8, size=n).round()
        vwap_noise = rng.normal(0, 0.004, size=n)
        daily_vwap = close * (1 + vwap_noise)
        turnover = daily_vwap * volume
        rows.append(pd.DataFrame({"date": dates, "ric": ric, "close": close, "volume": volume, "turnover": turnover, "daily_vwap": daily_vwap, "turnover_source": "MOCK_TURNOVER"}))
    return pd.concat(rows, ignore_index=True)


def generate_mock_static(rics: list[str]) -> pd.DataFrame:
    return pd.DataFrame({"ric": rics, "company_name": rics, "isin": pd.NA, "exchange_name": "MOCK", "currency": "JPY", "company_market_cap": np.nan})


def select_as_of_date(hist: pd.DataFrame, target_rics: list[str]) -> pd.Timestamp:
    if hist.empty:
        raise ValueError("history が空です。RIC、権限、日付範囲、フィールドを確認してください。")
    target = _as_date(AS_OF_DATE)
    if target is not None:
        candidates = hist.loc[hist["date"] <= target, "date"].dropna().sort_values().unique()
        if len(candidates) == 0:
            raise ValueError(f"AS_OF_DATE={target.date()} 以前のデータがありません。")
        return pd.Timestamp(candidates[-1]).normalize()

    coverage = hist.loc[hist["close"].notna()].groupby("date")["ric"].nunique().sort_index().div(max(1, len(set(target_rics))))
    good = coverage[coverage >= AS_OF_MIN_COVERAGE]
    if not good.empty:
        return pd.Timestamp(good.index[-1]).normalize()
    warnings.warn("AS_OF_MIN_COVERAGEを満たす日がありません。取得データの最大日付を使います。")
    return pd.Timestamp(hist["date"].max()).normalize()


def validate_history(hist: pd.DataFrame, required_rics: list[str], as_of_date: pd.Timestamp) -> pd.DataFrame:
    latest_by_ric = hist.groupby("ric")["date"].max() if not hist.empty else pd.Series(dtype="datetime64[ns]")
    missing_rics = sorted(set(required_rics) - set(latest_by_ric.index.astype(str)))
    rows = [
        {"metric": "required_ric_count", "value": len(required_rics)},
        {"metric": "history_available_ric_count", "value": int(hist["ric"].nunique()) if not hist.empty else 0},
        {"metric": "history_missing_ric_count", "value": len(missing_rics)},
        {"metric": "history_rows", "value": int(len(hist))},
        {"metric": "as_of_date", "value": str(as_of_date.date())},
        {"metric": "min_date", "value": str(hist["date"].min().date()) if not hist.empty else None},
        {"metric": "max_date", "value": str(hist["date"].max().date()) if not hist.empty else None},
        {"metric": "price_missing_pct", "value": float(hist["close"].isna().mean() * 100) if not hist.empty else np.nan},
        {"metric": "volume_missing_pct", "value": float(hist["volume"].isna().mean() * 100) if not hist.empty else np.nan},
        {"metric": "turnover_missing_pct", "value": float(hist["turnover"].isna().mean() * 100) if not hist.empty else np.nan},
        {"metric": "missing_rics", "value": ", ".join(missing_rics[:50]) + (" ..." if len(missing_rics) > 50 else "")},
    ]
    if "turnover_source" in hist.columns:
        rows.extend({"metric": f"turnover_source::{k}", "value": v} for k, v in hist["turnover_source"].value_counts(dropna=False).to_dict().items())
    if not hist.empty:
        vwap_ratio = hist["daily_vwap"] / hist["close"]
        suspicious = hist.loc[vwap_ratio.notna() & ((vwap_ratio < 0.5) | (vwap_ratio > 2.0))]
        rows.append({"metric": "suspicious_daily_vwap_ratio_count", "value": int(len(suspicious))})
    return pd.DataFrame(rows)


def compute_stock_avwap(hist: pd.DataFrame, anchor_date: str | pd.Timestamp) -> pd.DataFrame:
    anchor = _as_date(anchor_date)
    if anchor is None:
        raise ValueError("anchor_date を日付として解釈できません。")
    df = hist[hist["date"] >= anchor].copy()
    df = df.sort_values(["ric", "date"]).reset_index(drop=True)
    valid = df["volume"].notna() & df["turnover"].notna() & (df["volume"] > 0) & (df["turnover"] > 0)
    df["valid_avwap_input"] = valid
    df["turnover_for_cum"] = np.where(valid, df["turnover"], 0.0)
    df["volume_for_cum"] = np.where(valid, df["volume"], 0.0)
    df["cum_turnover"] = df.groupby("ric")["turnover_for_cum"].cumsum()
    df["cum_volume"] = df.groupby("ric")["volume_for_cum"].cumsum()
    df["avwap"] = np.where(df["cum_volume"] > 0, df["cum_turnover"] / df["cum_volume"], np.nan)
    df["valid_days_since_anchor"] = df.groupby("ric")["valid_avwap_input"].cumsum()
    df["avwap_quality_flag"] = np.where(df["valid_days_since_anchor"] < MIN_VALID_DAYS, "SHORT_HISTORY", "OK")
    return df.drop(columns=["turnover_for_cum", "volume_for_cum"])


def compute_stock_metrics_asof(avwap_hist: pd.DataFrame, as_of_date: pd.Timestamp, static: pd.DataFrame) -> pd.DataFrame:
    """Take last stock observation <= as_of_date, with stale-date control."""
    df = avwap_hist[avwap_hist["date"] <= as_of_date].copy()
    if df.empty:
        return pd.DataFrame()
    last = df.sort_values(["ric", "date"]).groupby("ric").tail(1).copy().reset_index(drop=True)
    last["stale_days"] = (as_of_date - last["date"]).dt.days
    stale = last["stale_days"] > MAX_STALE_DAYS_FOR_POSITION
    last.loc[stale, ["close", "avwap"]] = np.nan
    last["above_avwap"] = np.where(last["close"].notna() & last["avwap"].notna(), last["close"] > last["avwap"], pd.NA)
    last["avwap_gap_pct"] = np.where(last["close"].notna() & last["avwap"].notna() & (last["avwap"] != 0), 100.0 * (last["close"] / last["avwap"] - 1.0), np.nan)

    h = avwap_hist[avwap_hist["date"] <= as_of_date].sort_values(["ric", "date"])
    roll = h.groupby("ric", group_keys=False).tail(20).groupby("ric").agg(
        turnover_20d=("turnover", "sum"),
        turnover_5d=("turnover", lambda x: x.tail(5).sum()),
        turnover_last_day=("turnover", lambda x: x.tail(1).sum()),
    ).reset_index()
    last = last.merge(roll, on="ric", how="left")

    if static is not None and not static.empty:
        last = last.merge(static, on="ric", how="left")
    else:
        last["company_name"] = last["ric"]
    return last


In [ ]:
def _format_top_names(df: pd.DataFrame, metric: str, ascending: bool, n: int = 3) -> str:
    if metric not in df.columns:
        return ""
    tmp = df.dropna(subset=[metric]).copy()
    if tmp.empty:
        return ""
    name_col = "company_name" if "company_name" in tmp.columns else "ric"
    tmp[name_col] = tmp[name_col].fillna(tmp["ric"])
    tmp = tmp.sort_values(metric, ascending=ascending).head(n)
    return "; ".join(f"{row[name_col]}({row['ric']} {row[metric]:+.1f}%)" for _, row in tmp.iterrows())


def active_basket_rows(baskets: pd.DataFrame, on_date: pd.Timestamp) -> pd.DataFrame:
    df = baskets.copy()
    mask = pd.Series(True, index=df.index)
    mask &= df["valid_from"].isna() | (df["valid_from"] <= on_date)
    mask &= df["valid_to"].isna() | (df["valid_to"] >= on_date)
    return df.loc[mask].copy()


def compute_basket_position(baskets: pd.DataFrame, stock_asof: pd.DataFrame, as_of_date: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    active = active_basket_rows(baskets, as_of_date)
    merged = active.merge(stock_asof, on="ric", how="left", suffixes=("", "_stock"))
    merged["has_position_data"] = merged["close"].notna() & merged["avwap"].notna() & merged["above_avwap"].notna()
    merged["raw_weight"] = pd.to_numeric(merged["weight"], errors="coerce").fillna(0.0)

    total = merged.groupby("basket_id").agg(
        basket_name=("basket_name", "first"),
        basket_group=("basket_group", "first"),
        total_name_count=("ric", "nunique"),
        weight_sum=("raw_weight", "sum"),
    ).reset_index()

    valid = merged[merged["has_position_data"]].copy()
    if valid.empty:
        pos = total.copy()
        for col in ["valid_name_count", "valid_weight_sum", "weight_coverage_pct", "weighted_above_avwap_pct", "equal_weight_above_avwap_pct", "weighted_mean_avwap_gap_pct", "weighted_median_avwap_gap_pct", "equal_weight_mean_avwap_gap_pct", "top_positive_gap_names", "top_negative_gap_names"]:
            pos[col] = np.nan
        merged["normalized_weight"] = np.nan
        return pos, merged

    valid["valid_weight_sum"] = valid.groupby("basket_id")["raw_weight"].transform("sum")
    valid["normalized_weight"] = np.where(valid["valid_weight_sum"] > 0, valid["raw_weight"] / valid["valid_weight_sum"], np.nan)
    valid["above_float"] = valid["above_avwap"].astype(float)
    weight_sum_lookup = total.set_index("basket_id")["weight_sum"].to_dict()

    def agg_one(g: pd.DataFrame, basket_id: str) -> pd.Series:
        weight_sum_total = weight_sum_lookup.get(basket_id, np.nan)
        valid_weight_sum = g["raw_weight"].sum()
        return pd.Series({
            "valid_name_count": int(g["ric"].nunique()),
            "valid_weight_sum": float(valid_weight_sum),
            "weight_coverage_pct": 100.0 * safe_divide(valid_weight_sum, weight_sum_total),
            "weighted_above_avwap_pct": 100.0 * np.nansum(g["normalized_weight"] * g["above_float"]),
            "equal_weight_above_avwap_pct": 100.0 * g["above_float"].mean(),
            "weighted_mean_avwap_gap_pct": np.nansum(g["normalized_weight"] * g["avwap_gap_pct"]),
            "weighted_median_avwap_gap_pct": weighted_median(g["avwap_gap_pct"], g["normalized_weight"]),
            "equal_weight_mean_avwap_gap_pct": g["avwap_gap_pct"].mean(),
            "top_positive_gap_names": _format_top_names(g, "avwap_gap_pct", ascending=False, n=3),
            "top_negative_gap_names": _format_top_names(g, "avwap_gap_pct", ascending=True, n=3),
        })

    pos_rows = []
    for bid, g in valid.groupby("basket_id"):
        row = agg_one(g, bid)
        row["basket_id"] = bid
        pos_rows.append(row)
    pos_valid = pd.DataFrame(pos_rows)
    pos = total.merge(pos_valid, on="basket_id", how="left")
    merged = merged.merge(valid[["basket_id", "ric", "normalized_weight"]], on=["basket_id", "ric"], how="left")
    return pos, merged


def compute_basket_daily_turnover(baskets: pd.DataFrame, hist: pd.DataFrame) -> pd.DataFrame:
    h = hist[["date", "ric", "turnover"]].copy()
    h = h[h["turnover"].notna() & (h["turnover"] > 0)]
    b = baskets.copy()
    b["raw_weight"] = pd.to_numeric(b["weight"], errors="coerce").fillna(0.0)
    merged = b.merge(h, on="ric", how="inner")
    mask = pd.Series(True, index=merged.index)
    mask &= merged["valid_from"].isna() | (merged["valid_from"] <= merged["date"])
    mask &= merged["valid_to"].isna() | (merged["valid_to"] >= merged["date"])
    merged = merged.loc[mask].copy()
    if merged.empty:
        return pd.DataFrame(columns=["date", "basket_id", "effective_turnover", "flow_valid_name_count", "flow_valid_weight_sum", "top_turnover_contributors"])

    if FLOW_TURNOVER_MODE == "normalized_weighted":
        denom = merged.groupby(["date", "basket_id"])["raw_weight"].transform("sum")
        merged["flow_weight"] = np.where(denom > 0, merged["raw_weight"] / denom, np.nan)
    elif FLOW_TURNOVER_MODE == "raw_weighted":
        merged["flow_weight"] = merged["raw_weight"]
    elif FLOW_TURNOVER_MODE == "equal_member":
        merged["flow_weight"] = 1.0
    else:
        raise ValueError(f"Unsupported FLOW_TURNOVER_MODE: {FLOW_TURNOVER_MODE}")

    merged["effective_turnover_component"] = merged["flow_weight"] * merged["turnover"]

    def top_turnover(g: pd.DataFrame, n: int = 3) -> str:
        tmp = g.sort_values("effective_turnover_component", ascending=False).head(n)
        labels = []
        for _, row in tmp.iterrows():
            note = row.get("note", "")
            label = str(note) if pd.notna(note) and str(note).strip() else str(row["ric"])
            labels.append(f"{label}({row['ric']})")
        return "; ".join(labels)

    daily = merged.groupby(["date", "basket_id"]).agg(
        effective_turnover=("effective_turnover_component", "sum"),
        flow_valid_name_count=("ric", "nunique"),
        flow_valid_weight_sum=("raw_weight", "sum"),
    ).reset_index()
    top_rows = []
    for (d, bid), g in merged.groupby(["date", "basket_id"]):
        top_rows.append({"date": d, "basket_id": bid, "top_turnover_contributors": top_turnover(g)})
    top = pd.DataFrame(top_rows)
    return daily.merge(top, on=["date", "basket_id"], how="left").sort_values(["basket_id", "date"])


def compute_flow_denominator(hist: pd.DataFrame, baskets: pd.DataFrame) -> pd.DataFrame:
    if FLOW_DENOMINATOR_MODE == "none":
        return pd.DataFrame({"date": sorted(hist["date"].dropna().unique()), "denominator_turnover": np.nan})
    if FLOW_DENOMINATOR_MODE == "basket_union":
        denom_rics = set(to_list_unique(baskets["ric"]))
    elif FLOW_DENOMINATOR_MODE == "market_universe":
        denom_rics = set(to_list_unique(MARKET_UNIVERSE_RICS))
    else:
        raise ValueError(f"Unsupported FLOW_DENOMINATOR_MODE: {FLOW_DENOMINATOR_MODE}")
    d = hist[hist["ric"].isin(denom_rics) & hist["turnover"].notna() & (hist["turnover"] > 0)]
    return d.groupby("date", as_index=False).agg(denominator_turnover=("turnover", "sum"))


def get_window_dates(hist: pd.DataFrame, as_of_date: pd.Timestamp) -> tuple[list[pd.Timestamp], list[pd.Timestamp]]:
    dates = sorted(pd.Timestamp(x).normalize() for x in hist.loc[hist["date"] <= as_of_date, "date"].dropna().unique())
    need = FLOW_SHORT_WINDOW + (FLOW_BASE_WINDOW if FLOW_BASE_EXCLUDE_RECENT else 0)
    if len(dates) < need:
        warnings.warn(f"flow計算に必要な日数が不足している可能性があります。available_dates={len(dates)}, needed={need}")
    short_dates = dates[-FLOW_SHORT_WINDOW:]
    if FLOW_BASE_EXCLUDE_RECENT:
        base_dates = dates[-(FLOW_SHORT_WINDOW + FLOW_BASE_WINDOW) : -FLOW_SHORT_WINDOW]
    else:
        base_dates = dates[-FLOW_BASE_WINDOW:]
    return short_dates, base_dates


def compute_basket_flow(baskets: pd.DataFrame, hist: pd.DataFrame, as_of_date: pd.Timestamp) -> pd.DataFrame:
    daily = compute_basket_daily_turnover(baskets, hist)
    denom = compute_flow_denominator(hist, baskets)
    daily = daily.merge(denom, on="date", how="left")
    daily["turnover_share"] = daily["effective_turnover"] / daily["denominator_turnover"] if FLOW_DENOMINATOR_MODE != "none" else np.nan

    short_dates, base_dates = get_window_dates(hist, as_of_date)
    short_set, base_set = set(short_dates), set(base_dates)
    daily = daily.sort_values(["basket_id", "date"])

    short = daily[daily["date"].isin(short_set)].groupby("basket_id").agg(
        effective_turnover_short=("effective_turnover", "sum"),
        effective_turnover_last_day=("effective_turnover", lambda x: x.tail(1).sum()),
        short_valid_days=("date", "nunique"),
        top_turnover_contributors=("top_turnover_contributors", lambda x: x.dropna().iloc[-1] if len(x.dropna()) else ""),
    ).reset_index()
    base = daily[daily["date"].isin(base_set)].groupby("basket_id").agg(
        effective_turnover_base=("effective_turnover", "sum"),
        base_valid_days=("date", "nunique"),
    ).reset_index()

    denom_short = denom[denom["date"].isin(short_set)]["denominator_turnover"].sum()
    denom_base = denom[denom["date"].isin(base_set)]["denominator_turnover"].sum()

    all_baskets = baskets[["basket_id", "basket_name", "basket_group"]].drop_duplicates("basket_id")
    flow = all_baskets.merge(short, on="basket_id", how="left").merge(base, on="basket_id", how="left")
    for col in ["effective_turnover_short", "effective_turnover_last_day", "effective_turnover_base"]:
        flow[col] = pd.to_numeric(flow[col], errors="coerce")

    flow["avg_effective_turnover_short"] = flow["effective_turnover_short"] / max(1, len(short_dates))
    flow["avg_effective_turnover_base"] = flow["effective_turnover_base"] / max(1, len(base_dates))
    flow["flow_turnover_rel_change_pct"] = [pct_change_ratio(s, b) for s, b in zip(flow["avg_effective_turnover_short"], flow["avg_effective_turnover_base"])]

    if FLOW_DENOMINATOR_MODE != "none":
        flow["short_turnover_share_pct"] = 100.0 * flow["effective_turnover_short"] / denom_short if denom_short > 0 else np.nan
        flow["base_turnover_share_pct"] = 100.0 * flow["effective_turnover_base"] / denom_base if denom_base > 0 else np.nan
        flow["flow_share_rel_change_pct"] = [pct_change_ratio(s, b) for s, b in zip(flow["short_turnover_share_pct"], flow["base_turnover_share_pct"])]
    else:
        flow["short_turnover_share_pct"] = np.nan
        flow["base_turnover_share_pct"] = np.nan
        flow["flow_share_rel_change_pct"] = np.nan

    flow["short_window_start"] = min(short_dates) if short_dates else pd.NaT
    flow["short_window_end"] = max(short_dates) if short_dates else pd.NaT
    flow["base_window_start"] = min(base_dates) if base_dates else pd.NaT
    flow["base_window_end"] = max(base_dates) if base_dates else pd.NaT
    return flow


In [ ]:
def classify_basket_quadrants(metrics: pd.DataFrame, x_metric: str, y_metric: str) -> pd.DataFrame:
    df = metrics.copy()
    flow = pd.to_numeric(df[x_metric], errors="coerce")
    pos = pd.to_numeric(df[y_metric], errors="coerce")
    conditions = [
        flow.ge(0) & pos.ge(50),
        flow.lt(0) & pos.ge(50),
        flow.lt(0) & pos.lt(50),
        flow.ge(0) & pos.lt(50),
    ]
    choices = ["健全", "剥落注意", "投げ・底ばい", "底入れ候補"]
    df["quadrant"] = np.select(conditions, choices, default="判定不能")
    return df


def build_basket_metrics(baskets: pd.DataFrame, stock_asof: pd.DataFrame, hist: pd.DataFrame, as_of_date: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    pos, drill = compute_basket_position(baskets, stock_asof, as_of_date)
    flow = compute_basket_flow(baskets, hist, as_of_date)
    metrics = pos.merge(flow.drop(columns=[c for c in ["basket_name", "basket_group"] if c in flow.columns]), on="basket_id", how="outer")

    metrics["as_of_date"] = as_of_date
    metrics["position_metric"] = "weighted_above_avwap_pct" if POSITION_WEIGHT_MODE == "portfolio_weight" else "equal_weight_above_avwap_pct"
    metrics["x_metric"] = PLOT_X_METRIC if not (FLOW_DENOMINATOR_MODE == "none" and PLOT_X_METRIC == "flow_share_rel_change_pct") else "flow_turnover_rel_change_pct"

    x_metric_actual = metrics["x_metric"].iloc[0]
    y_metric_actual = metrics["position_metric"].iloc[0]
    metrics = classify_basket_quadrants(metrics, x_metric_actual, y_metric_actual)

    def flags(row: pd.Series) -> str:
        out = []
        if pd.isna(row.get("weight_coverage_pct")) or row.get("weight_coverage_pct", 0) < 100.0 * MIN_WEIGHT_COVERAGE:
            out.append("LOW_WEIGHT_COVERAGE")
        if pd.isna(row.get("valid_name_count")) or row.get("valid_name_count", 0) < MIN_BASKET_NAMES:
            out.append("FEW_VALID_NAMES")
        if pd.isna(row.get("weighted_above_avwap_pct")):
            out.append("MISSING_AVWAP")
        if pd.isna(row.get("flow_turnover_rel_change_pct")):
            out.append("MISSING_TURNOVER_FLOW")
        return "OK" if not out else "|".join(out)

    metrics["data_quality_flag"] = metrics.apply(flags, axis=1)

    preferred_cols = [
        "as_of_date", "basket_id", "basket_name", "basket_group",
        "total_name_count", "valid_name_count", "weight_sum", "valid_weight_sum", "weight_coverage_pct",
        "weighted_above_avwap_pct", "equal_weight_above_avwap_pct",
        "weighted_mean_avwap_gap_pct", "weighted_median_avwap_gap_pct", "equal_weight_mean_avwap_gap_pct",
        "effective_turnover_short", "effective_turnover_base", "effective_turnover_last_day",
        "avg_effective_turnover_short", "avg_effective_turnover_base", "flow_turnover_rel_change_pct",
        "short_turnover_share_pct", "base_turnover_share_pct", "flow_share_rel_change_pct",
        "quadrant", "data_quality_flag", "top_positive_gap_names", "top_negative_gap_names", "top_turnover_contributors",
        "short_window_start", "short_window_end", "base_window_start", "base_window_end", "position_metric", "x_metric",
    ]
    for col in preferred_cols:
        if col not in metrics.columns:
            metrics[col] = np.nan
    metrics = metrics[preferred_cols]

    if "normalized_weight" not in drill.columns:
        drill["normalized_weight"] = np.nan
    drill["effective_turnover_5d_component"] = pd.to_numeric(drill.get("normalized_weight", np.nan), errors="coerce") * pd.to_numeric(drill.get("turnover_5d", np.nan), errors="coerce")
    drill["turnover_contribution_pct"] = np.nan
    total_component = drill.groupby("basket_id")["effective_turnover_5d_component"].transform("sum")
    ok = total_component > 0
    drill.loc[ok, "turnover_contribution_pct"] = 100.0 * drill.loc[ok, "effective_turnover_5d_component"] / total_component[ok]
    return metrics, drill


def plot_basket_flow_position(metrics: pd.DataFrame) -> go.Figure:
    if metrics.empty:
        raise ValueError("basket_metrics が空です。")
    x_metric = metrics["x_metric"].dropna().iloc[0] if "x_metric" in metrics.columns and metrics["x_metric"].notna().any() else PLOT_X_METRIC
    y_metric = metrics["position_metric"].dropna().iloc[0] if "position_metric" in metrics.columns and metrics["position_metric"].notna().any() else "weighted_above_avwap_pct"
    size_col = "short_turnover_share_pct" if metrics["short_turnover_share_pct"].notna().any() else "effective_turnover_short"

    plot_df = metrics.copy()
    plot_df["label"] = plot_df["basket_name"].fillna(plot_df["basket_id"])
    plot_df["size_for_plot"] = pd.to_numeric(plot_df[size_col], errors="coerce")
    if plot_df["size_for_plot"].isna().all() or (plot_df["size_for_plot"].fillna(0) <= 0).all():
        plot_df["size_for_plot"] = 1.0

    hover_cols = [
        "basket_id", "basket_name", "basket_group", "total_name_count", "valid_name_count", "weight_coverage_pct",
        "weighted_above_avwap_pct", "equal_weight_above_avwap_pct", "weighted_mean_avwap_gap_pct",
        "flow_share_rel_change_pct", "flow_turnover_rel_change_pct", "short_turnover_share_pct", "base_turnover_share_pct",
        "quadrant", "data_quality_flag", "top_turnover_contributors", "top_positive_gap_names", "top_negative_gap_names",
    ]
    hover_cols = [c for c in hover_cols if c in plot_df.columns]

    fig = px.scatter(
        plot_df,
        x=x_metric,
        y=y_metric,
        size="size_for_plot",
        color="basket_group",
        text="label",
        hover_data=hover_cols,
        title=(
            f"Theme Basket flow × position | as of {pd.Timestamp(plot_df['as_of_date'].iloc[0]).date()} | "
            f"anchor {ANCHOR_DATE} | short={FLOW_SHORT_WINDOW}d base={FLOW_BASE_WINDOW}d"
        ),
    )
    fig.update_traces(textposition="top center")
    fig.add_vline(x=0, line_dash="dash")
    fig.add_hline(y=50, line_dash="dash")

    x_vals = pd.to_numeric(plot_df[x_metric], errors="coerce")
    y_vals = pd.to_numeric(plot_df[y_metric], errors="coerce")
    x_min = min(-10.0, float(np.nanmin(x_vals)) if x_vals.notna().any() else -10.0)
    x_max = max(10.0, float(np.nanmax(x_vals)) if x_vals.notna().any() else 10.0)
    y_min = min(0.0, float(np.nanmin(y_vals)) if y_vals.notna().any() else 0.0)
    y_max = max(100.0, float(np.nanmax(y_vals)) if y_vals.notna().any() else 100.0)
    x_pad = max(5.0, 0.12 * (x_max - x_min))
    y_pad = max(5.0, 0.08 * (y_max - y_min))
    fig.update_xaxes(range=[x_min - x_pad, x_max + x_pad], title=x_metric)
    fig.update_yaxes(range=[max(-5, y_min - y_pad), min(105, y_max + y_pad)], title=y_metric)

    for x, y, text in [(x_max, 95, "健全"), (x_min, 95, "剥落注意"), (x_min, 5, "投げ・底ばい"), (x_max, 5, "底入れ候補")]:
        fig.add_annotation(x=x, y=y, text=text, showarrow=False, opacity=0.65)
    fig.update_layout(height=720, legend_title_text="basket_group")
    return fig


def show_quadrant_rankings(metrics: pd.DataFrame) -> None:
    x_metric = metrics["x_metric"].dropna().iloc[0] if "x_metric" in metrics.columns and metrics["x_metric"].notna().any() else PLOT_X_METRIC
    y_metric = metrics["position_metric"].dropna().iloc[0] if "position_metric" in metrics.columns and metrics["position_metric"].notna().any() else "weighted_above_avwap_pct"
    view_cols = ["basket_name", "basket_group", "quadrant", x_metric, y_metric, "weighted_mean_avwap_gap_pct", "short_turnover_share_pct", "data_quality_flag"]
    view_cols = [c for c in view_cols if c in metrics.columns]
    configs = [
        ("健全", [x_metric, y_metric], [False, False]),
        ("底入れ候補", [x_metric, "weighted_mean_avwap_gap_pct"], [False, False]),
        ("剥落注意", [x_metric, "short_turnover_share_pct"], [True, False]),
        ("投げ・底ばい", [y_metric, x_metric], [True, True]),
        ("判定不能", ["basket_name"], [True]),
    ]
    for quadrant, sort_cols, ascending in configs:
        sub = metrics[metrics["quadrant"].eq(quadrant)].copy()
        if sub.empty:
            continue
        sort_cols = [c for c in sort_cols if c in sub.columns]
        print(f"\n## {quadrant}")
        display(sub.sort_values(sort_cols, ascending=ascending[: len(sort_cols)])[view_cols].reset_index(drop=True))


def show_basket_drilldown(drilldown: pd.DataFrame, basket_id: str, sort_by: str = "normalized_weight", ascending: bool = False) -> pd.DataFrame:
    sub = drilldown[drilldown["basket_id"].eq(basket_id)].copy()
    if sub.empty:
        print(f"basket_id={basket_id!r} に該当する行がありません。")
        return sub
    if sort_by in sub.columns:
        sub = sub.sort_values(sort_by, ascending=ascending)
    cols = [
        "basket_id", "basket_name", "ric", "company_name", "note", "weight", "normalized_weight",
        "date", "close", "avwap", "above_avwap", "avwap_gap_pct", "turnover_last_day", "turnover_5d", "turnover_20d",
        "turnover_contribution_pct", "valid_days_since_anchor", "turnover_source", "avwap_quality_flag", "stale_days",
    ]
    cols = [c for c in cols if c in sub.columns]
    display(sub[cols].reset_index(drop=True))
    return sub


def build_data_quality_report(baskets: pd.DataFrame, hist: pd.DataFrame, stock_asof: pd.DataFrame, basket_metrics: pd.DataFrame, retrieval_failures: list[dict[str, Any]], as_of_date: pd.Timestamp, required_rics: list[str]) -> pd.DataFrame:
    rows = [
        {"metric": "input_basket_count", "value": baskets["basket_id"].nunique()},
        {"metric": "input_basket_ric_rows", "value": len(baskets)},
        {"metric": "unique_required_ric_count", "value": len(required_rics)},
        {"metric": "history_rows", "value": len(hist)},
        {"metric": "history_available_ric_count", "value": hist["ric"].nunique() if not hist.empty else 0},
        {"metric": "stock_metrics_asof_count", "value": len(stock_asof)},
        {"metric": "basket_metrics_count", "value": len(basket_metrics)},
        {"metric": "as_of_date", "value": str(as_of_date.date())},
        {"metric": "retrieval_failure_count", "value": len(retrieval_failures)},
    ]
    if not basket_metrics.empty:
        rows.append({"metric": "low_weight_coverage_baskets", "value": int((basket_metrics["weight_coverage_pct"] < 100.0 * MIN_WEIGHT_COVERAGE).sum())})
        rows.append({"metric": "few_valid_names_baskets", "value": int((basket_metrics["valid_name_count"] < MIN_BASKET_NAMES).sum())})
        rows.append({"metric": "missing_flow_baskets", "value": int(basket_metrics["flow_turnover_rel_change_pct"].isna().sum())})
        rows.append({"metric": "missing_position_baskets", "value": int(basket_metrics["weighted_above_avwap_pct"].isna().sum())})
        rows.extend({"metric": f"data_quality_flag::{k}", "value": v} for k, v in basket_metrics["data_quality_flag"].value_counts(dropna=False).to_dict().items())
    if not hist.empty:
        rows.append({"metric": "close_missing_pct", "value": float(hist["close"].isna().mean() * 100)})
        rows.append({"metric": "volume_missing_pct", "value": float(hist["volume"].isna().mean() * 100)})
        rows.append({"metric": "turnover_missing_pct", "value": float(hist["turnover"].isna().mean() * 100)})
        rows.extend({"metric": f"turnover_source::{k}", "value": v} for k, v in hist["turnover_source"].value_counts(dropna=False).to_dict().items())
    return pd.DataFrame(rows)


def export_outputs(basket_metrics: pd.DataFrame, stock_metrics: pd.DataFrame, drilldown: pd.DataFrame, data_quality: pd.DataFrame, fig: Optional[go.Figure], as_of_date: pd.Timestamp) -> dict[str, Path]:
    stamp = pd.Timestamp(as_of_date).strftime("%Y%m%d")
    paths = {
        "basket_metrics": OUTPUT_DIR / f"basket_metrics_{stamp}.csv",
        "stock_metrics": OUTPUT_DIR / f"stock_metrics_{stamp}.csv",
        "basket_drilldown": OUTPUT_DIR / f"basket_drilldown_{stamp}.csv",
        "data_quality": OUTPUT_DIR / f"data_quality_{stamp}.csv",
    }
    basket_metrics.to_csv(paths["basket_metrics"], index=False, encoding="utf-8-sig")
    stock_metrics.to_csv(paths["stock_metrics"], index=False, encoding="utf-8-sig")
    drilldown.to_csv(paths["basket_drilldown"], index=False, encoding="utf-8-sig")
    data_quality.to_csv(paths["data_quality"], index=False, encoding="utf-8-sig")
    if fig is not None:
        html_path = OUTPUT_DIR / f"basket_flow_position_{stamp}.html"
        fig.write_html(html_path)
        paths["figure_html"] = html_path
    return paths


## 8. Run pipeline

このセルを実行すると、LSEGからデータを取得し、AVWAP・flow・positionを計算して散布図とランキングを表示します。

In [ ]:
ld = None
session = None
retrieval_failures: list[dict[str, Any]] = []

try:
    required_rics = get_required_rics(basket_constituents)
    basket_rics = to_list_unique(basket_constituents["ric"])
    history_start, history_end = determine_history_dates()

    print(f"Required RICs: {len(required_rics):,}")
    print(f"History window: {history_start.date()} to {history_end.date()}")
    print(f"Anchor date: {ANCHOR_DATE}")

    if RUN_WITH_MOCK_DATA:
        print("RUN_WITH_MOCK_DATA=True: LSEGには接続せず、サンプルデータで計算します。")
        static_metadata = generate_mock_static(required_rics)
        history = generate_mock_history(required_rics, history_start, history_end)
    else:
        ld, session = open_lseg_session()
        static_metadata, static_failures = fetch_static_metadata(ld, required_rics)
        retrieval_failures.extend(static_failures)
        history, history_failures = fetch_history_batched(ld, required_rics, history_start, history_end)
        retrieval_failures.extend(history_failures)

    as_of_date = select_as_of_date(history[history["ric"].isin(basket_rics)], basket_rics)
    print(f"Selected as_of_date: {as_of_date.date()}")

    history_quality = validate_history(history, required_rics, as_of_date)
    print("\nHistory quality")
    display(history_quality)

    stock_avwap_history = compute_stock_avwap(history[history["ric"].isin(basket_rics)], ANCHOR_DATE)
    stock_metrics = compute_stock_metrics_asof(stock_avwap_history, as_of_date, static_metadata)

    basket_metrics, basket_drilldown = build_basket_metrics(basket_constituents, stock_metrics, history, as_of_date)

    data_quality_report = build_data_quality_report(basket_constituents, history, stock_metrics, basket_metrics, retrieval_failures, as_of_date, required_rics)

    print("\nBasket metrics")
    display(basket_metrics)

    fig = plot_basket_flow_position(basket_metrics)
    if os.getenv("SUPPRESS_FIG_SHOW", "0") != "1":
        fig.show()

    print("\nQuadrant rankings")
    show_quadrant_rankings(basket_metrics)

    print("\nData quality report")
    display(data_quality_report)

    export_paths = export_outputs(basket_metrics, stock_metrics, basket_drilldown, data_quality_report, fig, as_of_date)
    print("\nExported files")
    for k, v in export_paths.items():
        print(f"{k}: {v}")

finally:
    if AUTO_CLOSE_SESSION and (ld is not None) and not RUN_WITH_MOCK_DATA:
        close_lseg_session(ld)


## 9. Drilldown

`basket_id` を指定すると、構成銘柄別のウェイト、AVWAP乖離、商い寄与を確認できます。

In [ ]:
# 例: "semis_ai", "banks", "power_heavy"
SELECTED_BASKET_ID = basket_constituents["basket_id"].iloc[0]
_ = show_basket_drilldown(basket_drilldown, SELECTED_BASKET_ID, sort_by="normalized_weight", ascending=False)


## 10. Optional: inspect retrieval failures

In [ ]:
retrieval_failures_df = pd.DataFrame(retrieval_failures)
display(retrieval_failures_df if not retrieval_failures_df.empty else pd.DataFrame([{"status": "OK", "message": "No retrieval failures recorded"}]))
